# Geometric Regeneration Experiment with nanoGPT

**Hypothesis:** Revisiting earlier tokens using forward+backward context at geometrically spaced intervals (multiples of 32) improves downstream next-token prediction accuracy.

**Regeneration Schedule:**
- At 64 tokens generated: regenerate [0, 32) using forward context [32, 64)
- At 128 tokens: regenerate [32, 64) using forward context [64, 128)
- At 256 tokens: regenerate [64, 128) using forward context [128, 256)

**Runtime:** Colab Pro with A100 GPU recommended. Total ~4-6 hours.

## 1. Setup: Clone nanoGPT and install dependencies

In [1]:
!git clone https://github.com/karpathy/nanoGPT.git
%cd nanoGPT
!pip install torch numpy transformers datasets tiktoken tqdm scipy

Cloning into 'nanoGPT'...
remote: Enumerating objects: 689, done.
remote: Total 689 (delta 0), reused 0 (delta 0), pack-reused 689 (from 1)
Receiving objects: 100% (689/689), 975.25 KiB | 7.07 MiB/s, done.
Resolving deltas: 100% (382/382), done.
/content/nanoGPT


## 2. Check GPU

In [2]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Memory: {mem_gb:.1f} GB")
print(f"bfloat16 support: {torch.cuda.is_bf16_supported()}")

# Determine config adjustments based on GPU
if mem_gb >= 35:
    print("\n-> A100 detected. Default config will work.")
    BATCH_SIZE = 16
    GRAD_ACCUM = 8
elif mem_gb >= 14:
    print("\n-> V100/T4 detected. Will use reduced batch size.")
    BATCH_SIZE = 4
    GRAD_ACCUM = 32
else:
    print("\n-> Small GPU. This may be very slow.")
    BATCH_SIZE = 2
    GRAD_ACCUM = 64

print(f"Batch size: {BATCH_SIZE}, Gradient accumulation: {GRAD_ACCUM}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

GPU: Tesla T4
Memory: 15.6 GB
bfloat16 support: True

-> V100/T4 detected. Will use reduced batch size.
Batch size: 4, Gradient accumulation: 32
Effective batch size: 128


## 3. Prepare SimpleWiki data

In [3]:
import os
import re
import bz2
import requests
import numpy as np
import tiktoken
from tqdm import tqdm

DATA_DIR = "data/simplewiki"
os.makedirs(DATA_DIR, exist_ok=True)

# Download
url = "https://dumps.wikimedia.org/simplewiki/latest/simplewiki-latest-pages-articles.xml.bz2"
filepath = os.path.join(DATA_DIR, "simplewiki.xml.bz2")

if not os.path.exists(filepath):
    print(f"Downloading SimpleWiki ...")
    resp = requests.get(url, stream=True)
    total = int(resp.headers.get("content-length", 0))
    with open(filepath, "wb") as f:
        with tqdm(total=total, unit="B", unit_scale=True) as pbar:
            for chunk in resp.iter_content(8192):
                f.write(chunk)
                pbar.update(len(chunk))
else:
    print("Already downloaded.")

100%|██████████| 345M/345M [01:37<00:00, 3.55MB/s]


In [4]:
def clean_wikitext(text):
    text = re.sub(r"\{\{[^}]*\}\}", "", text)
    text = re.sub(r"<ref[^>]*>.*?</ref>", "", text, flags=re.DOTALL)
    text = re.sub(r"<ref[^/]*/>", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\[\[[^|\]]*\|([^\]]*)\]\]", r"\1", text)
    text = re.sub(r"\[\[([^\]]*)\]\]", r"\1", text)
    text = re.sub(r"\[https?://[^\s\]]+\s*([^\]]*)\]", r"\1", text)
    text = re.sub(r"\[\[Category:[^\]]*\]\]", "", text)
    text = re.sub(r"'{2,5}", "", text)
    text = re.sub(r"={2,6}\s*(.*?)\s*={2,6}", r"\n\1\n", text)
    text = re.sub(r"^\*+\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"^#+\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()

# Extract articles
print("Extracting articles ...")
articles = []
current_text = []
in_text = False

with bz2.open(filepath, "rt", encoding="utf-8") as f:
    for line in tqdm(f, desc="Parsing"):
        if "<text" in line:
            in_text = True
            m = re.search(r"<text[^>]*>(.*)", line)
            if m:
                current_text.append(m.group(1))
        elif "</text>" in line:
            in_text = False
            m = re.search(r"(.*)</text>", line)
            if m:
                current_text.append(m.group(1))
            raw = "\n".join(current_text).strip()
            if len(raw) > 100:
                cleaned = clean_wikitext(raw)
                if len(cleaned) > 100:
                    articles.append(cleaned)
            current_text = []
        elif in_text:
            current_text.append(line.strip())

print(f"Extracted {len(articles)} articles")

Extracting articles ...


Parsing: 30373577it [03:19, 152020.98it/s]

Extracted 341066 articles


In [5]:
# Tokenize and save
enc = tiktoken.get_encoding("gpt2")
print("Tokenizing ...")

all_tokens = []
for art in tqdm(articles, desc="Tokenize"):
    tokens = enc.encode_ordinary(art)
    tokens.append(enc.eot_token)
    all_tokens.extend(tokens)

all_tokens = np.array(all_tokens, dtype=np.uint16)
print(f"Total tokens: {len(all_tokens):,}")

# 90/10 split
n_train = int(len(all_tokens) * 0.9)
train_tokens = all_tokens[:n_train]
val_tokens = all_tokens[n_train:]

train_tokens.tofile(os.path.join(DATA_DIR, "train.bin"))
val_tokens.tofile(os.path.join(DATA_DIR, "val.bin"))
print(f"Train: {len(train_tokens):,} tokens")
print(f"Val:   {len(val_tokens):,} tokens")

Tokenizing ...


Tokenize: 100%|██████████| 341066/341066 [02:14<00:00, 2544.99it/s]


Total tokens: 256,819,201
Train: 231,137,280 tokens
Val:   25,681,921 tokens


## 4. Patch nanoGPT's model.py for custom attention masks

This adds a `custom_mask` parameter that flows through GPT -> Block -> CausalSelfAttention. When `custom_mask=None` (default), behavior is identical to the original. When provided, it replaces the causal mask, enabling bidirectional attention during regeneration.

In [ ]:
import shutil

model_path = "model.py"

# Backup
shutil.copy(model_path, "model_original.py")
print("Backed up model.py -> model_original.py")

with open(model_path, "r") as f:
    code = f.read()

# PATCH 1: CausalSelfAttention.forward() signature
code = code.replace(
    "    def forward(self, x):\n        B, T, C = x.size()",
    "    def forward(self, x, custom_mask=None):\n        B, T, C = x.size()"
)

# PATCH 2: Skip flash attention when custom_mask is provided
code = code.replace(
    "        if self.flash:\n",
    "        if self.flash and custom_mask is None:\n"
)

# PATCH 3: Use custom_mask instead of causal mask when provided
old_mask = "            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))"
new_mask = """            if custom_mask is not None:
                att = att.masked_fill(custom_mask == 0, float('-inf'))
            else:
                att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))"""
code = code.replace(old_mask, new_mask)

# PATCH 4: Block.forward() passes custom_mask
code = code.replace(
    "    def forward(self, x):\n        x = x + self.attn(self.ln_1(x))",
    "    def forward(self, x, custom_mask=None):\n        x = x + self.attn(self.ln_1(x), custom_mask=custom_mask)"
)

# PATCH 5: GPT.forward() accepts and propagates custom_mask
code = code.replace(
    "    def forward(self, idx, targets=None):",
    "    def forward(self, idx, targets=None, custom_mask=None):"
)
code = code.replace(
    "        for block in self.transformer.h:\n            x = block(x)",
    "        for block in self.transformer.h:\n            x = block(x, custom_mask=custom_mask)"
)

with open(model_path, "w") as f:
    f.write(code)

print("Patched model.py successfully!")

In [ ]:
# Verify the patch
from model import GPTConfig, GPT

config = GPTConfig(
    n_layer=2, n_head=2, n_embd=64,
    block_size=32, vocab_size=50257, dropout=0.0, bias=True
)
test_model = GPT(config).to("cuda")

x = torch.randint(0, 50257, (1, 16), device="cuda")

# Normal forward
logits1, _ = test_model(x)

# Forward with explicit causal mask (should match)
T = 16
causal = torch.tril(torch.ones(T, T, device="cuda")).unsqueeze(0).unsqueeze(0)
logits2, _ = test_model(x, custom_mask=causal)

diff = (logits1 - logits2).abs().max().item()
print(f"Max diff between normal and explicit causal mask: {diff:.8f}")
assert diff < 1e-4, "PATCH FAILED: outputs differ!"
print("PATCH VERIFIED: custom_mask works correctly.")

del test_model
torch.cuda.empty_cache()

## 5. Write training config

In [ ]:
os.makedirs("config", exist_ok=True)

config_text = f"""# Training config for GPT-2 124M on SimpleWiki
out_dir = 'out-simplewiki'
eval_interval = 500
log_interval = 50
eval_iters = 200
eval_only = False
always_save_checkpoint = True

dataset = 'simplewiki'
gradient_accumulation_steps = {GRAD_ACCUM}
batch_size = {BATCH_SIZE}

n_layer = 12
n_head = 12
n_embd = 768
block_size = 256
dropout = 0.1
bias = True

learning_rate = 6e-4
max_iters = 30000
weight_decay = 0.1
beta1 = 0.9
beta2 = 0.95
grad_clip = 1.0

decay_lr = True
warmup_iters = 2000
lr_decay_iters = 30000
min_lr = 6e-5

device = 'cuda'
dtype = 'bfloat16'
compile = True
init_from = 'scratch'
"""

with open("config/train_simplewiki.py", "w") as f:
    f.write(config_text)

print(f"Config written with batch_size={BATCH_SIZE}, grad_accum={GRAD_ACCUM}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

## 6. Train the model

This is the longest step. Expected training time:
- A100: ~3-4 hours
- V100: ~8-12 hours
- T4: ~15+ hours (consider reducing max_iters)

Watch for the validation loss to plateau. You can interrupt early if it stops improving.

In [ ]:
!python train.py config/train_simplewiki.py

## 7. Sanity check: generate some text

In [ ]:
import torch
import tiktoken
from model import GPTConfig, GPT

# Load checkpoint
ckpt = torch.load("out-simplewiki/ckpt.pt", map_location="cuda")
gptconf = GPTConfig(**ckpt["model_args"])
model = GPT(gptconf)
state_dict = ckpt["model"]
for k in list(state_dict.keys()):
    if k.startswith("_orig_mod."):
        state_dict[k[len("_orig_mod."):]] = state_dict.pop(k)
model.load_state_dict(state_dict)
model.to("cuda")
model.eval()

enc = tiktoken.get_encoding("gpt2")
prompt = "The history of mathematics"
tokens = enc.encode(prompt)
x = torch.tensor([tokens], dtype=torch.long, device="cuda")

with torch.no_grad():
    out = model.generate(x, max_new_tokens=100, temperature=0.8, top_k=40)
print(enc.decode(out[0].tolist()))

del model
torch.cuda.empty_cache()

## 8. Evaluation: Baseline vs Geometric Regeneration

This is the core experiment. We compare:
- **Baseline:** standard autoregressive generation (greedy, model consumes its own outputs)
- **Regeneration:** same initial generation, then geometric regeneration passes with custom attention mask

Both use free generation (not teacher-forced), so regeneration can actually affect downstream predictions.

We measure accuracy on tokens AFTER each regeneration boundary.

In [ ]:
import torch
import numpy as np
from contextlib import nullcontext
from scipy import stats
from model import GPTConfig, GPT

# ============================================================
# Configuration
# ============================================================
OUT_DIR = "out-simplewiki"
DATA_DIR = "data/simplewiki"
NUM_SAMPLES = 500       # number of sequences to evaluate
NUM_GENERATE = 256      # tokens to generate per sample
PROMPT_LEN = 32         # ground-truth tokens used as prompt
REGEN_BLOCK = 32        # regeneration block size
DEVICE = "cuda"
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Load model
print(f"Loading model from {OUT_DIR} ...")
ckpt = torch.load(os.path.join(OUT_DIR, "ckpt.pt"), map_location=DEVICE)
gptconf = GPTConfig(**ckpt["model_args"])
model = GPT(gptconf)
state_dict = ckpt["model"]
for k in list(state_dict.keys()):
    if k.startswith("_orig_mod."):
        state_dict[k[len("_orig_mod."):]] = state_dict.pop(k)
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()
print(f"Loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

# Load validation data
val_data = np.memmap(os.path.join(DATA_DIR, "val.bin"), dtype=np.uint16, mode="r")
print(f"Validation data: {len(val_data):,} tokens")

# Setup dtype context
dtype = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
ptdtype = {"bfloat16": torch.bfloat16, "float16": torch.float16}[dtype]
ctx = torch.amp.autocast(device_type=DEVICE, dtype=ptdtype)

In [ ]:
# ============================================================
# Core functions
# ============================================================

def build_regen_mask(seq_len, regen_start, regen_end, device):
    """
    Build attention mask for regeneration.

    Tokens in [regen_start, regen_end) can attend to:
      - All tokens before them (causal within regen block)
      - All tokens in [regen_end, seq_len) (forward context)
    Tokens outside the regen window keep standard causal attention.
    """
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    for i in range(regen_start, regen_end):
        mask[i, regen_end:seq_len] = 1.0
    return mask.unsqueeze(0).unsqueeze(0)


def get_regen_schedule(total_tokens, block_size=32):
    """
    Geometric regeneration schedule.
    Returns list of (trigger_point, regen_start, regen_end).
    """
    schedule = []
    trigger = block_size * 2
    regen_start = 0
    while trigger <= total_tokens:
        regen_end = regen_start + block_size
        schedule.append((trigger, regen_start, regen_end))
        regen_start = regen_end
        trigger *= 2
    return schedule


@torch.no_grad()
def generate_baseline(model, prompt, num_generate, ctx):
    """Standard greedy autoregressive generation."""
    idx = prompt.clone()
    for _ in range(num_generate):
        idx_cond = idx if idx.size(1) <= model.config.block_size else idx[:, -model.config.block_size:]
        with ctx:
            logits, _ = model(idx_cond)
        next_tok = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        idx = torch.cat([idx, next_tok], dim=1)
    return idx[:, prompt.size(1):]


@torch.no_grad()
def generate_with_regen(model, prompt, num_generate, ctx, block_size=32):
    """
    Generation with geometric regeneration passes.

    1. Generate all tokens autoregressively (same as baseline)
    2. Apply regeneration at geometric boundaries
    """
    prompt_len = prompt.size(1)
    idx = prompt.clone()

    # Initial generation (identical to baseline)
    for _ in range(num_generate):
        idx_cond = idx if idx.size(1) <= model.config.block_size else idx[:, -model.config.block_size:]
        with ctx:
            logits, _ = model(idx_cond)
        next_tok = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        idx = torch.cat([idx, next_tok], dim=1)

    # Apply geometric regeneration
    schedule = get_regen_schedule(num_generate, block_size)
    regen_boundaries = []

    for trigger, regen_start_gen, regen_end_gen in schedule:
        regen_start = prompt_len + regen_start_gen
        regen_end = prompt_len + regen_end_gen

        if regen_end > idx.size(1):
            break

        regen_boundaries.append(regen_end_gen)

        # Context window for regeneration
        context_end = min(prompt_len + trigger, idx.size(1))
        context_start = max(0, context_end - model.config.block_size)
        context = idx[:, context_start:context_end].clone()

        local_regen_start = regen_start - context_start
        local_regen_end = regen_end - context_start
        local_seq_len = context.size(1)

        # Build custom mask
        mask = build_regen_mask(
            local_seq_len, local_regen_start, local_regen_end, idx.device
        )

        # Regenerate left-to-right within the block
        for pos in range(local_regen_start, local_regen_end):
            with ctx:
                logits, _ = model(context, custom_mask=mask)
            new_token = logits[:, pos, :].argmax(dim=-1)
            context[:, pos] = new_token

        # Write back
        idx[:, regen_start:regen_end] = context[:, local_regen_start:local_regen_end]

    return idx[:, prompt_len:], regen_boundaries


print("Functions defined.")

# Show the regeneration schedule
schedule = get_regen_schedule(NUM_GENERATE, REGEN_BLOCK)
print(f"\nRegeneration schedule for {NUM_GENERATE} generated tokens:")
for trigger, rs, re_ in schedule:
    print(f"  At token {trigger}: regenerate [{rs}, {re_}) with forward context [{re_}, {trigger})")

In [ ]:
# ============================================================
# Run evaluation
# ============================================================

total_seq_len = PROMPT_LEN + NUM_GENERATE
assert total_seq_len <= gptconf.block_size, \
    f"prompt + generation ({total_seq_len}) exceeds block_size ({gptconf.block_size})"

baseline_accs = []
regen_accs = []
baseline_post_regen = {re_: [] for _, _, re_ in schedule}
regen_post_regen = {re_: [] for _, _, re_ in schedule}

print(f"Evaluating {NUM_SAMPLES} samples ...")
print(f"Prompt: {PROMPT_LEN} tokens, Generate: {NUM_GENERATE} tokens")
print("-" * 60)

for i in range(NUM_SAMPLES):
    # Random starting point in validation data
    start = np.random.randint(0, len(val_data) - total_seq_len - 1)
    ground_truth = torch.from_numpy(
        val_data[start:start + total_seq_len].astype(np.int64)
    ).to(DEVICE)

    prompt = ground_truth[:PROMPT_LEN].unsqueeze(0)
    gt_gen = ground_truth[PROMPT_LEN:]

    # Baseline
    bl_gen = generate_baseline(model, prompt, NUM_GENERATE, ctx).squeeze(0)

    # Regeneration
    rg_gen, boundaries = generate_with_regen(
        model, prompt, NUM_GENERATE, ctx, block_size=REGEN_BLOCK
    )
    rg_gen = rg_gen.squeeze(0)

    # Overall accuracy
    bl_acc = (bl_gen == gt_gen).float().mean().item()
    rg_acc = (rg_gen == gt_gen).float().mean().item()
    baseline_accs.append(bl_acc)
    regen_accs.append(rg_acc)

    # Per-region: tokens AFTER each regen boundary
    for _, _, re_ in schedule:
        if re_ < NUM_GENERATE:
            bl_r = (bl_gen[re_:] == gt_gen[re_:]).float().mean().item()
            rg_r = (rg_gen[re_:] == gt_gen[re_:]).float().mean().item()
            baseline_post_regen[re_].append(bl_r)
            regen_post_regen[re_].append(rg_r)

    if (i + 1) % 50 == 0:
        bl_m = np.mean(baseline_accs[-50:])
        rg_m = np.mean(regen_accs[-50:])
        print(f"  [{i+1:>4d}/{NUM_SAMPLES}] baseline={bl_m:.4f}  regen={rg_m:.4f}  delta={rg_m-bl_m:+.4f}")

print("\nDone!")

## 9. Results

In [ ]:
print("=" * 60)
print("RESULTS")
print("=" * 60)

bl_mean, bl_std = np.mean(baseline_accs), np.std(baseline_accs)
rg_mean, rg_std = np.mean(regen_accs), np.std(regen_accs)
delta = rg_mean - bl_mean

print(f"\nOverall next-token accuracy (free generation, {NUM_SAMPLES} samples):")
print(f"  Baseline:      {bl_mean:.4f} +/- {bl_std:.4f}")
print(f"  Regeneration:  {rg_mean:.4f} +/- {rg_std:.4f}")
print(f"  Delta:         {delta:+.4f}")

# Paired t-test
t_stat, p_val = stats.ttest_rel(regen_accs, baseline_accs)
print(f"\n  Paired t-test: t={t_stat:.3f}, p={p_val:.6f}")
print(f"  {'Statistically significant (p < 0.05)' if p_val < 0.05 else 'NOT significant (p >= 0.05)'}")

print(f"\nAccuracy on tokens AFTER each regeneration boundary:")
for _, rs, re_ in schedule:
    if re_ in baseline_post_regen and len(baseline_post_regen[re_]) > 0:
        bl_r = np.mean(baseline_post_regen[re_])
        rg_r = np.mean(regen_post_regen[re_])
        d = rg_r - bl_r
        print(f"  After regen [{rs},{re_}): baseline={bl_r:.4f}, regen={rg_r:.4f}, delta={d:+.4f}")

In [ ]:
# Save results
results = {
    "baseline_accs": baseline_accs,
    "regen_accs": regen_accs,
    "baseline_post_regen": dict(baseline_post_regen),
    "regen_post_regen": dict(regen_post_regen),
    "schedule": schedule,
    "config": {
        "num_samples": NUM_SAMPLES,
        "num_generate": NUM_GENERATE,
        "prompt_len": PROMPT_LEN,
        "regen_block": REGEN_BLOCK,
    }
}
torch.save(results, os.path.join(OUT_DIR, "regen_eval_results.pt"))
print("Results saved.")

## 10. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Distribution of per-sample accuracy differences
diffs = [r - b for r, b in zip(regen_accs, baseline_accs)]
axes[0].hist(diffs, bins=50, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5, label="No change")
axes[0].axvline(np.mean(diffs), color="green", linestyle="--", linewidth=1.5,
                label=f"Mean: {np.mean(diffs):+.4f}")
axes[0].set_xlabel("Accuracy difference (regen - baseline)")
axes[0].set_ylabel("Count")
axes[0].set_title("Per-sample accuracy change")
axes[0].legend()

# Plot 2: Per-region accuracy comparison
regions = []
bl_means = []
rg_means = []
for _, rs, re_ in schedule:
    if re_ in baseline_post_regen and len(baseline_post_regen[re_]) > 0:
        regions.append(f"After [{rs},{re_})")
        bl_means.append(np.mean(baseline_post_regen[re_]))
        rg_means.append(np.mean(regen_post_regen[re_]))

x_pos = np.arange(len(regions))
w = 0.35
axes[1].bar(x_pos - w/2, bl_means, w, label="Baseline", color="lightcoral", edgecolor="black")
axes[1].bar(x_pos + w/2, rg_means, w, label="Regeneration", color="steelblue", edgecolor="black")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(regions, rotation=25, ha="right")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy after regen boundaries")
axes[1].legend()

# Plot 3: Cumulative accuracy over generation position
# Show how accuracy degrades over position for both methods
# (computed from stored per-region data)
deltas = []
labels = []
for _, rs, re_ in schedule:
    if re_ in baseline_post_regen and len(baseline_post_regen[re_]) > 0:
        d = np.mean(regen_post_regen[re_]) - np.mean(baseline_post_regen[re_])
        deltas.append(d)
        labels.append(f"[{rs},{re_})")

colors = ["green" if d > 0 else "red" for d in deltas]
axes[2].bar(range(len(deltas)), deltas, color=colors, edgecolor="black", alpha=0.7)
axes[2].set_xticks(range(len(labels)))
axes[2].set_xticklabels(labels, rotation=25, ha="right")
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set_ylabel("Accuracy delta (regen - baseline)")
axes[2].set_title("Regeneration effect by boundary")

plt.tight_layout()
plt.savefig("regen_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to regen_results.png")

## 11. Interpretation Notes

**If regeneration helps (positive delta):**
This is a surprisingly strong result, because the model was trained only with causal attention. It never saw forward context during training, yet it can use it to improve predictions. This suggests the attention mechanism can generalize to unseen mask patterns.

**If regeneration hurts or shows no effect (negative/zero delta):**
This is the expected outcome given the distribution mismatch. The model's attention weights were learned under causal masking, so revealing future tokens introduces an input distribution the model was never trained on. To properly test the hypothesis, you would need to include regeneration passes during training.

**Next steps if results are promising:**
1. Add regeneration to the training loop so the model learns both attention patterns
2. Vary the regeneration block size (16, 32, 64) and geometric schedule
3. Test on LAMBADA or HellaSwag for downstream task impact
4. Compare with the paper's AGR approach